#### **Schema Validation**
1. What is Schema-on-read/ Schema-on-write?
2. Column order Validation
3. Data Type Validation
4. Column Name Validation
5. Nullability Validation
6. Extra Columns Validation

In [0]:
%sql
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_sv
(
  customer_id INT NOT NULL,
  invoice_no STRING,
  quantity INT,
  price FLOAT,
  invoice_date DATE
);

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
SELECT customer_id, invoice_no, quantity, price , invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_sv
limit 5;

In [0]:
%sql
SELECT MIN(customer_id), MAX(Customer_id), count(*) as countOfRecords
from deltacatalog.deltadb.invoices_sv;

#### Scenario 1: Column Order Validation

- Since Quantity and Customer_id are both INT, lets try exchanging them.

In [0]:
%sql
select customer_id, invoice_no, quantity, price, invoice_date
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

In [0]:
%sql
select quantity, invoice_no, customer_id, price, invoice_date
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
select quantity, invoice_no, customer_id, price, invoice_date
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

- Column matching by position happens instead of Name.
- This corrupts the data.
- Never use INSERT unless very sure of positioning of columns.

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_sv
where customer_id=99999;

In [0]:
%sql
SELECT quantity, invoice_no, customer_id, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id DESC LIMIT 5;

In [0]:
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT quantity, invoice_no, customer_id, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id DESC LIMIT 5
) src
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT *


In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_sv
where customer_id > 100

- MERGE RAN absolutely fine, it was able to identify which column to merge/insert. Even though order was disturbed.
- INSERT Matches column by position.
- MERGE Matches column by name. It finds right name, even if they are not ordered correctly.

#### Scenario 2 : Data Type Validation

- DataType Mismatch for customer_id
- Error : The value 'ABC' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 1, position 1) ==
INSERT INTO deltacatalog.deltadb.invoices_sv

In [0]:
%skip
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES
(
  'ABC',
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES
(
  '99499',
  'I45678', 
  10,
  98.75,
  '2025-01-01'
)

- Here Spark tries to convert String input to Int type and this time conversion was successful so without throwing error, it inserted the data

In [0]:
%sql
Select * from deltacatalog.deltadb.invoices_sv where customer_id = 99499;

#### Scenario 3 : Column Name Validation

In [0]:
%sql
-- INSERT INTO deltacatalog.deltadb.invoices_sv
select customer_id, invoice_no, quantity, price, invoice_date
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
select customer_id as c_id, invoice_no, quantity, price, invoice_date
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

- Insert column matching by position, name does not matter.

In [0]:
%sql
Select * from deltacatalog.deltadb.invoices_sv where customer_id=99999;

In [0]:
%sql
SELECT customer_id, invoice_no, quantity, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id LIMIT 5

- Here changing the column names while reading from new data, insert will be an issue in merge now, bcos in merge insert happens only by column name mapping not by position mapping.
- Error : Cannot resolve customer_id in INSERT clause given columns src.c_id, src.invoice_no, src.qty, src.price, src.invoice_date. SQLSTATE: 42601
- Solution : **INSERT (customer_id, invoice_no, quantity, price, invoice_date) VALUES (src.c_id, src.invoice_no, src.qty, src.price, current_date())**

In [0]:
%skip
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT customer_id as c_id, invoice_no, quantity as qty, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id LIMIT 5
) src
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT customer_id as c_id, invoice_no, quantity as qty, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id LIMIT 5
) src
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT (customer_id, invoice_no, quantity, price, invoice_date) VALUES (src.c_id, src.invoice_no, src.qty, src.price, current_date())

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_sv
where customer_id > 100

- Run Again for Updates this time

In [0]:
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT customer_id as c_id, invoice_no, quantity as qty, CAST(price as FLOAT) , invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY Customer_id LIMIT 5
) src
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT (customer_id, invoice_no, quantity, price, invoice_date) VALUES (src.c_id, src.invoice_no, src.qty, src.price, current_date())

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_sv
where customer_id > 100

#### Scenario 4: Nullability Validation

- Below query will not run, bcos **not null constraint** gets violated for customer_id.
- NOT NULL constraint violated for column: customer_id.
 SQLSTATE: 23502

In [0]:
%skip
%sql
INSERT into deltacatalog.deltadb.invoices_sv
VALUES 
(
  NULL,
  NULL,
  NULL,
  NULL,
  NUll
)

- This works with a random number in customer_id column.

In [0]:
%sql
INSERT into deltacatalog.deltadb.invoices_sv
VALUES 
(
  78912,
  NULL,
  NULL,
  NULL,
  NUll
)

#### Scenario 5 : Extra Column Validation

- Changing the column number, adding one extra
- Error : A metadata mismatch was detected when writing to the Delta table. SQLSTATE: 42KDG

In [0]:
%skip
%sql
INSERT INTO deltacatalog.deltadb.invoices_sv
select customer_id, invoice_no, quantity , price, invoice_date, "VIP" as customer_type
from VALUES(99999,'I12345',10,100,'2025-01-01') 
AS T(customer_id, invoice_no, quantity, price , invoice_date)

In [0]:
%sql
SELECT customer_id as c_id, invoice_no, quantity as qty, CAST(price as FLOAT) , invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
where customer_id BETWEEN 150 and 155

In [0]:
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT customer_id, invoice_no, quantity, CAST(price as FLOAT) , invoice_date, "VIP" as customer_type
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  where customer_id BETWEEN 150 and 155
) src
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT *

- Merge is quite robust, searches by name if not able to find one will not bother to insert.

In [0]:
%sql
SELECT customer_id, invoice_no, quantity, CAST(price as FLOAT) , invoice_date
FROM deltacatalog.deltadb.invoices_sv
where customer_id BETWEEN 150 and 155

In [0]:
%sql
MERGE INTO deltacatalog.deltadb.invoices_sv as tgt
USING (
  SELECT customer_id, invoice_no, quantity, CAST(price as FLOAT) , invoice_date, "VIP" as customer_type
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  where customer_id BETWEEN 150 and 155
) src
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date()
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT customer_id, invoice_no, quantity, CAST(price as FLOAT) , invoice_date
FROM deltacatalog.deltadb.invoices_sv
where customer_id BETWEEN 150 and 155